In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Wed Feb  4 16:56:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 0,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

data_id = 'jxie/coco_captions'
data_split = 'train'
data_size = 1000
dataset = load_hf_dataset(data_id, split=data_split, train_size=data_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

In [4]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

if 'url' in dataset.features:
    dataset = dataset.map(process_image_with_url, num_proc=4) # Load and process images from URLs
else:
    dataset = dataset.map(process_image, num_proc=4) # Resize to (512, 512) and convert to RGB

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# Remove the original 'image' column
train_dataset = dataset.remove_columns('image')

# Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

In [6]:
from huggingface_hub import hf_hub_download

K = 8192

codebook_path = hf_hub_download(
    repo_id='alxxtexxr/VRBI-Dataset',
    filename=f'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook_faiss.{K}.npy',
    repo_type='dataset',
)

print("Codebook downloaded to:", codebook_path)

Codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--VRBI-Dataset/snapshots/4dbe255ac6935e9f0f8ab2964f60c7627c7b6eba/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook_faiss.8192.npy


In [7]:
%%capture
%pip install faiss-gpu-cu12

In [8]:
import numpy as np
import faiss

codebook = np.load(codebook_path).astype(np.float32) # (K, 2048) -> current: (16_384, 2048)
faiss.normalize_L2(codebook)

print("Codebook shape:", codebook.shape)
print("Mean L2 norm (should be ~1):", np.linalg.norm(codebook, axis=1).mean())

Codebook shape: (8192, 2048)
Mean L2 norm (should be ~1): 1.0


In [ ]:
d = codebook.shape[1] # Current: 2048

# FAISS search (L2 is equivalent to cosine for unit vectors)
index = faiss.IndexFlatL2(d)
index.add(codebook)

In [10]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = 'alxxtexxr/VRBI-Dataset'
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2


In [17]:
import numpy as np
import faiss

# Load all batch file paths
batch_paths = sorted([f for f in vision_data_dir.glob('*.npz')])
batch_size = len(train_dataset) // len(batch_paths) # Current: 100

distances = []
codebook_idxs = []

for batch_path in batch_paths:
    batch = np.load(batch_path, mmap_mode='r')
    visual_embs_i = batch['visual_embs'].astype(np.float32) # Current: (32_400, 2048)
    
    faiss.normalize_L2(visual_embs_i)
    
    distances_i, codebook_idxs_i = index.search(visual_embs_i, 1)
    codebook_idxs_i = codebook_idxs_i.squeeze()
    
    distances_i = distances_i.flatten()                       # Current: (32400,)
    codebook_idxs_i = codebook_idxs_i.reshape(batch_size, -1) # Current: (100, 324)
    
    print(f"Distances shape: {distances_i.shape}, type: {type(distances_i)}")
    print(f"Codebook indices shape: {codebook_idxs_i.shape}, type: {type(codebook_idxs_i)}")
    print()
    
    distances.extend(distances_i)
    codebook_idxs.extend(codebook_idxs_i)

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Distance

In [12]:
# Compute RMSE
rmse = np.sqrt(np.vstack(distances).mean())
print("RMSE:", rmse)

RMSE: 0.6418361


In [18]:
codebook_idxs[0]

array([2365, 5532, 6739, 7513, 7270, 5520, 5334, 4884, 1587, 3680,  461,
       8069,  351, 6150, 6150,  236, 6150, 2947, 2302,  893, 1314, 1314,
       7405, 5911, 5898, 5991, 2998, 1583, 2352, 8069, 4980, 4980, 4980,
       4980, 4980, 6150, 1652, 4611, 2366, 2366, 7405, 3044, 7353, 3044,
       5001, 8157, 6793,  682,  351, 4980, 4980, 4980, 4980, 4980, 4460,
       5334, 5334, 6525, 4724, 5334, 5334,  245, 3044, 4884, 6993, 6793,
       2612, 4797, 1236, 2590,  803, 1236,   44,  819, 5546, 4980, 7405,
       4071, 6993, 4008, 4316, 4884, 3044, 3127, 4331, 3948, 3948, 2121,
       3948, 2121, 6993, 1370, 2366, 6525, 7405, 2840, 6993, 3044, 3044,
       5407, 2352, 6822, 8069, 6993, 3948, 3948, 3948, 2177, 5334, 2366,
       8137, 7612, 2366, 2840, 5243, 3044, 2129, 5338, 7893, 8069, 5334,
       8069, 4071, 4422, 4318, 2281, 6947, 2366, 2366, 5546, 4724, 5334,
       3370, 5334, 6522, 7263,  788, 8069, 8069, 1096, 4530, 6165, 4422,
       2735, 2366, 2366, 2366, 2425, 6525, 4071, 69

In [36]:
def serialize_img_data(example, idx):
    codebook_idxs_i = codebook_idxs[idx].tolist()
    image_serialized = "".join([f'<|vtoken_{i}|>' for i in codebook_idxs_i])
    example['image_serialized'] = image_serialized
    example['codebook_idxs'] = codebook_idxs_i
    return example

dataset_serialized = train_dataset.map(serialize_img_data, with_indices=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [39]:
dataset_serialized.push_to_hub('alxxtexxr/coco_captions_1k_serialized')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          |  524kB /  390MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/coco_captions_1k_serialized/commit/b5b3ec57c21a4af8dd4f788452cd938c49a5d397', commit_message='Upload dataset', commit_description='', oid='b5b3ec57c21a4af8dd4f788452cd938c49a5d397', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/coco_captions_1k_serialized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/coco_captions_1k_serialized'), pr_revision=None, pr_num=None)